# Ensemble weight search
Поиск весов по совместимым OOF validation-предсказаниям.

In [ ]:
REPO_URL = "https://github.com/frest1ler/text-orientation-classification.git"
BRANCH = "main"
PROJECT_DIR = "/content/drive/MyDrive/text-orientation"
CANDIDATES = {
    "mobilenet_v3_large_robust": f"{PROJECT_DIR}/training/runs/robust/mobilenet_v3_large/full",
    "vit_b_16": f"{PROJECT_DIR}/training/runs/vit_b_16/full",
}
WEIGHT_STEP = 0.05
PROMOTE_ENSEMBLE = True
RUN_TESTS = False

In [ ]:
import os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
repo = Path("/content/text-orientation-classification")
if not (repo / ".git").is_dir(): subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if RUN_TESTS: subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
for selector, path in CANDIDATES.items():
    missing = [name for name in ("best.pt", "calibration.json", "validation_predictions.npz") if not (Path(path) / name).is_file()]
    if missing: raise FileNotFoundError(f"{selector}: missing {missing} in {path}")

In [ ]:
command = [sys.executable, "-m", "scripts.search_ensemble", "--project-dir", PROJECT_DIR, "--weight-step", str(WEIGHT_STEP)]
for selector, path in CANDIDATES.items(): command += ["--candidate", f"{selector}={path}"]
if not PROMOTE_ENSEMBLE: command.append("--no-promote")
subprocess.run(command, check=True)
print("Report:", Path(PROJECT_DIR) / "evaluation/ensembles/search.json")